# 📊 Phase 3: Dataset Pipeline

This notebook handles the dataset preparation workflow. It is driven dynamically by the parameters set in the YAML configuration to ensure reusability across datasets.

## Workflow Steps:
1. Define `PROJECT_ROOT` and verify required project folders
2. Load configurations from `qlora_config.yaml`
3. Read raw input CSV from `dataset.raw_path`
4. Validate that the dataset matches configured columns mapping
5. Clean strings, remove nulls, trim spaces, and remove duplicate records
6. Print comprehensive dataset split statistics
7. Split the data into **Train** and **Validation** sets based on the `dataset.test_split_ratio`
8. Format the subsets into standard ChatML conversation structures (`messages` list) using the configured system prompt
9. Save outputs to `data/processed/train.jsonl` and `data/processed/val.jsonl`
10. Verify the first two JSONL records and print total counts written to disk

## 1. Define PROJECT_ROOT and Verify Workspace Directories
We mount Google Drive (if on Google Colab) and verify that all necessary folders exist under our `PROJECT_ROOT`.

In [1]:
from pathlib import Path
import sys

# Determine if running on Google Colab
IN_COLAB = 'google.colab' in sys.modules

# Define PROJECT_ROOT exactly once
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path("/content/drive/MyDrive/LLM-Studio") if IN_COLAB else Path("../../").resolve()

if IN_COLAB:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

print(f"PROJECT_ROOT resolved to: {PROJECT_ROOT}")

# Verify required directories (Requirement 5)
required_dirs = [
    PROJECT_ROOT / "training",
    PROJECT_ROOT / "training/configs",
    PROJECT_ROOT / "training/scripts",
    PROJECT_ROOT / "data/raw",
    PROJECT_ROOT / "data/processed",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "models/adapters",
    PROJECT_ROOT / "models/merged"
]

print("\nAuditing required project directories...")
for folder in required_dirs:
    if not folder.exists():
        raise FileNotFoundError(
            f"Missing required project directory: {folder.relative_to(PROJECT_ROOT) if PROJECT_ROOT in folder.parents else folder}\n"
            f"Please ensure the project is correctly set up under PROJECT_ROOT: {PROJECT_ROOT}"
        )
    print(f"  ✓ {folder.relative_to(PROJECT_ROOT) if PROJECT_ROOT in folder.parents else folder} is present.")
print("✓ Project structure verification complete.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT resolved to: /content/drive/MyDrive/LLM-Studio

Auditing required project directories...
  ✓ training is present.
  ✓ training/configs is present.
  ✓ training/scripts is present.
  ✓ data/raw is present.
  ✓ data/processed is present.
  ✓ models is present.
  ✓ models/adapters is present.
  ✓ models/merged is present.
✓ Project structure verification complete.


## 2. Load Configuration
We load the YAML config file directly using our verified `PROJECT_ROOT` paths.

In [2]:
import yaml
import json

CONFIG_DIR = PROJECT_ROOT / "training/configs"
config_path = CONFIG_DIR / "qlora_config.yaml"

print(f"Loading config from: {config_path}")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

dataset_cfg = config["dataset"]
print(json.dumps(dataset_cfg, indent=2))

Loading config from: /content/drive/MyDrive/LLM-Studio/training/configs/qlora_config.yaml
{
  "raw_path": "data/raw/sample_dataset.csv",
  "train_path": "data/processed/train.jsonl",
  "val_path": "data/processed/val.jsonl",
  "test_split_ratio": 0.2,
  "random_seed": 42,
  "system_prompt": "You are a helpful assistant.",
  "columns": {
    "instruction": "instruction",
    "input": "input",
    "output": "output"
  }
}


## 3. Load Raw CSV Dataset
We load the configured CSV dataset. The path is derived directly from `PROJECT_ROOT`.

In [3]:
raw_path = PROJECT_ROOT / dataset_cfg["raw_path"]
if not raw_path.exists():
    raise FileNotFoundError(f"Raw dataset CSV not found at: {raw_path}")

print(f"Reading raw dataset CSV: {raw_path}")
df = pd.read_csv(raw_path) if 'pd' in globals() else None
if df is None:
    import pandas as pd
    df = pd.read_csv(raw_path)

print(f"Total rows loaded: {len(df)}")
print("Sample raw data preview:")
print(df.head(2))

Reading raw dataset CSV: /content/drive/MyDrive/LLM-Studio/data/raw/sample_dataset.csv
Total rows loaded: 10
Sample raw data preview:
                                         instruction  \
0  Explain what a large language model is in one ...   
1  Translate the following English phrase into Fr...   

                                input  \
0                                 NaN   
1  Thank you very much for your help.   

                                              output  
0  A large language model is an AI system trained...  
1                    Merci beaucoup pour votre aide.  


## 4. Validate, Clean, & De-duplicate Dataset
We verify that configured columns are present, clean missing elements, and remove duplicate records before partitioning splits.

In [4]:
cols = dataset_cfg["columns"]
inst_col = cols.get("instruction", "instruction")
input_col = cols.get("input", "input")
out_col = cols.get("output", "output")

# 1. Verify column presence
for c_name, actual_c in [("instruction", inst_col), ("output", out_col)]:
    if actual_c not in df.columns:
        raise ValueError(f"Mapped column '{c_name}' -> '{actual_c}' not found in CSV columns: {df.columns.tolist()}")

# 2. Clean null rows on essential parameters
initial_rows = len(df)
clean_df = df.dropna(subset=[inst_col, out_col]).copy()
rows_after_dropna = len(clean_df)

clean_df[inst_col] = clean_df[inst_col].astype(str).str.strip()
clean_df[out_col] = clean_df[out_col].astype(str).str.strip()

if input_col in df.columns:
    clean_df[input_col] = clean_df[input_col].fillna("").astype(str).str.strip()
else:
    clean_df[input_col] = ""

# 3. Remove duplicate records
dedup_columns = [inst_col, out_col]
if input_col in df.columns and input_col:
    dedup_columns.append(input_col)
dedup_df = clean_df.drop_duplicates(subset=dedup_columns).copy()
duplicates_removed = len(clean_df) - len(dedup_df)

print("--- Data Cleaning and De-duplication Summary ---")
print(f"Initial CSV rows:           {initial_rows}")
print(f"Rows after null dropping:   {rows_after_dropna} (Dropped {initial_rows - rows_after_dropna} rows)")
print(f"Duplicate records removed:  {duplicates_removed}")
print(f"Final clean dataset size:   {len(dedup_df)}")

--- Data Cleaning and De-duplication Summary ---
Initial CSV rows:           10
Rows after null dropping:   10 (Dropped 0 rows)
Duplicate records removed:  0
Final clean dataset size:   10


## 5. Train-Test Splits
We split the cleaned dataset into training and validation splits.

In [5]:
test_size = dataset_cfg.get("test_split_ratio", 0.2)
seed = dataset_cfg.get("random_seed", 42)

train_df, val_df = train_test_split(dedup_df, test_size=test_size, random_state=seed) if 'train_test_split' in globals() else (None, None)
if train_df is None:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(dedup_df, test_size=test_size, random_state=seed)

print("--- Dataset Partitioning Stats ---")
print(f"Train split size:       {len(train_df)} ({100*(1-test_size):.0f}%)")
print(f"Validation split size:  {len(val_df)} ({100*test_size:.0f}%)")

--- Dataset Partitioning Stats ---
Train split size:       8 (80%)
Validation split size:  2 (20%)


## 6. Format to ChatML and Save JSONL
We format each record as a conversational messages list containing system, user, and assistant turns using the system prompt configured in the YAML setting.

In [6]:
sys_prompt = dataset_cfg.get("system_prompt", "You are a helpful assistant.")
print(f"Using system prompt: '{sys_prompt}'")

def save_as_chatml_jsonl(dataframe, output_filepath_rel):
    out_path = PROJECT_ROOT / output_filepath_rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    
    records_written = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in dataframe.iterrows():
            instruction = row[inst_col]
            context = row.get(input_col, "")
            output = row[out_col]
            
            user_message = instruction
            if context:
                user_message += f"\n\nContext:\n{context}"
                
            record = {
                "messages": [
                    {"role": "system", "content": sys_prompt},
                    {"role": "user", "content": user_message},
                    {"role": "assistant", "content": output}
                ]
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            records_written += 1
    print(f"Saved {records_written} records to: {out_path.resolve()}")
    return records_written

total_train = save_as_chatml_jsonl(train_df, dataset_cfg["train_path"])
total_val = save_as_chatml_jsonl(val_df, dataset_cfg["val_path"])

print(f"\n✓ Successfully wrote a grand total of {total_train + total_val} JSONL records to disk.")

Using system prompt: 'You are a helpful assistant.'
Saved 8 records to: /content/drive/MyDrive/LLM-Studio/data/processed/train.jsonl
Saved 2 records to: /content/drive/MyDrive/LLM-Studio/data/processed/val.jsonl

✓ Successfully wrote a grand total of 10 JSONL records to disk.


## 7. Verify Processed Files
Quick sanity check of the saved JSONL structure. We verify and print **at least the first two JSONL records** to guarantee formatting compliance.

In [7]:
train_jsonl_path = PROJECT_ROOT / dataset_cfg["train_path"]
print(f"Reading generated file: {train_jsonl_path.resolve()}\n")

with open(train_jsonl_path, "r", encoding="utf-8") as f:
    # Read first record
    rec1 = json.loads(f.readline())
    # Read second record
    rec2 = json.loads(f.readline())
    
    print("--- Record #1 Verification Preview ---")
    print(json.dumps(rec1, indent=2))
    
    print("\n--- Record #2 Verification Preview ---")
    print(json.dumps(rec2, indent=2))

Reading generated file: /content/drive/MyDrive/LLM-Studio/data/processed/train.jsonl

--- Record #1 Verification Preview ---
{
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful assistant."
    },
    {
      "role": "user",
      "content": "Write a Python one-liner to check if a word is a palindrome."
    },
    {
      "role": "assistant",
      "content": "`is_palindrome = lambda word: word == word[::-1]`"
    }
  ]
}

--- Record #2 Verification Preview ---
{
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful assistant."
    },
    {
      "role": "user",
      "content": "Explain what a large language model is in one simple sentence."
    },
    {
      "role": "assistant",
      "content": "A large language model is an AI system trained on massive amounts of text to understand, generate, and predict human-like language."
    }
  ]
}
